In [ ]:
import os
os.environ['PYTHONWARNINGS'] = 'ignore'
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import decoupler as dc
import enrichmap as em
import gseapy as gp
import anndata
import matplotlib.pyplot as plt
import seaborn as sns
import time

from scipy.sparse import issparse, csr_matrix
from scipy.stats import spearmanr
from scipy.spatial.distance import cdist
from tqdm import tqdm

In [ ]:
sc.set_figure_params(frameon=False, fontsize=8)

# repository root (the folder containing `notebooks/`); safe to re-run this cell
project_dir = os.getcwd()
if os.path.basename(project_dir) == "notebooks":
    project_dir = os.path.dirname(project_dir)
project_dir += "/"
os.chdir(project_dir)
figure_dir = project_dir + 'figures/'
results_dir = project_dir + 'results/'
os.makedirs(figure_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

## 1. Visium simulation

Simulates a 10x Visium dataset matching the platform specifications:
- **Square capture area** with hexagonal spot grid at 100 μm centre-to-centre spacing
- **Variable spot count** (1500–4000 per slide, drawn randomly per seed) to mimic variable tissue coverage
- Spatially coherent domains via Voronoi tessellation from seed points
- Negative binomial count data with gene-specific dispersion and library size variation
- Signature gene set with graded fold changes
- Zero-inflation modelling dropout

In [ ]:
def simulate_visium(
    seed=0,
    n_genes=3000,
    n_signature_genes=20,
    n_clusters=4,
    target_cluster=1,
    n_spots_range=(1500, 4000),
    spot_spacing_um=100,
    spot_diameter_um=55,
    mean_counts_per_spot=5000,
    dispersion_range=(0.1, 2.0),
    fold_change_range=(1.5, 5.0),
    dropout_rate=0.1,
):
    """
    Simulate a 10x Visium spatial transcriptomics dataset.

    The grid layout matches the Visium platform: a square capture area
    with hexagonally arranged spots at 100 um centre-to-centre spacing.
    The number of spots is drawn randomly from ``n_spots_range`` to
    mimic variable tissue coverage across real slides. This is achieved
    by varying the capture area dimensions per simulation.

    Returns (adata, sig_genes, fold_changes, target_label).
    """
    rng = np.random.default_rng(seed)

    # ── Variable spot count via variable capture area ────────────
    # Draw a target spot count, then compute the grid side that
    # produces approximately that many spots in a hexagonal grid.
    # For a hex grid: n_spots ≈ n_rows × n_cols ≈ side^2 / 0.866
    n_spots_target = rng.integers(
        n_spots_range[0], n_spots_range[1] + 1,
    )
    grid_side = int(np.ceil(np.sqrt(n_spots_target * 0.866)))
    capture_area_um = grid_side * spot_spacing_um

    # ── Hexagonal grid within square capture area ────────────────
    n_cols = int(capture_area_um / spot_spacing_um)
    n_rows = int(capture_area_um / (spot_spacing_um * np.sqrt(3) / 2))

    x_coords, y_coords = [], []
    for row in range(n_rows):
        for col in range(n_cols):
            x = col * spot_spacing_um + (row % 2) * (spot_spacing_um / 2)
            y = row * spot_spacing_um * np.sqrt(3) / 2
            if x <= capture_area_um and y <= capture_area_um:
                x_coords.append(x)
                y_coords.append(y)

    x_coords = np.array(x_coords)
    y_coords = np.array(y_coords)
    n_spots = len(x_coords)
    coords = np.column_stack([x_coords, y_coords])

    # ── Spatially coherent clusters via Voronoi tessellation ─────
    margin = capture_area_um * 0.15
    seed_coords = np.column_stack([
        rng.uniform(margin, capture_area_um - margin, n_clusters),
        rng.uniform(margin, capture_area_um - margin, n_clusters),
    ])

    dists = cdist(coords, seed_coords)
    dists += rng.normal(0, spot_spacing_um * 0.8, dists.shape)
    cluster_ids = np.argmin(dists, axis=1)
    cluster_labels = np.array([f'cluster_{i+1}' for i in cluster_ids])

    target_label = f'cluster_{target_cluster + 1}'

    # ── Gene names ──────────────────────────────────────────────
    gene_names = [f'GENE_{i+1:05d}' for i in range(n_genes)]

    # ── Signature genes with graded fold changes ────────────────
    sig_indices = rng.choice(n_genes, n_signature_genes, replace=False)
    fold_changes = rng.uniform(
        fold_change_range[0], fold_change_range[1], n_signature_genes,
    )
    order = np.argsort(-fold_changes)
    sig_indices = sig_indices[order]
    fold_changes = fold_changes[order]
    sig_genes = [gene_names[i] for i in sig_indices]

    # ── Expression matrix (negative binomial) ───────────────────
    gene_means = np.exp(rng.normal(1.5, 1.0, n_genes))
    gene_means = np.clip(gene_means, 0.1, 100)

    dispersions = rng.uniform(
        dispersion_range[0], dispersion_range[1], n_genes,
    )

    lib_size_factors = np.exp(rng.normal(0, 0.3, n_spots))
    lib_size_factors *= mean_counts_per_spot / (
        gene_means.sum() * lib_size_factors.mean()
    )

    X = np.zeros((n_spots, n_genes), dtype=np.float32)
    target_mask = cluster_labels == target_label

    for g in range(n_genes):
        mu = gene_means[g] * lib_size_factors

        if g in sig_indices:
            fc_idx = np.where(sig_indices == g)[0][0]
            mu[target_mask] *= fold_changes[fc_idx]

        r = dispersions[g]
        p = r / (r + mu)
        counts = rng.negative_binomial(r, p)

        dropout_mask = rng.random(n_spots) < dropout_rate
        counts[dropout_mask] = 0

        X[:, g] = counts

    # ── Assemble AnnData ────────────────────────────────────────
    adata = anndata.AnnData(X=csr_matrix(X))
    adata.var_names = gene_names
    adata.obs_names = [f'spot_{i}' for i in range(n_spots)]
    adata.obs['cluster'] = pd.Categorical(cluster_labels)
    adata.obsm['spatial'] = coords

    adata.raw = adata.copy()

    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    return adata, sig_genes, fold_changes, target_label

## 2. Sanity check on one simulation

In [ ]:
adata, sig_genes, fold_changes, target_label = simulate_visium(seed=0)

print(f'Spots: {adata.n_obs}, Genes: {adata.n_vars}')
print(f'Clusters: {adata.obs["cluster"].value_counts().to_dict()}')
print(f'Target: {target_label}')
print(f'Signature: {len(sig_genes)} genes')
print(f'Fold changes: {fold_changes.round(2)}')

# Show a few seeds to demonstrate variable spot counts
print('\nSpot count variation across seeds:')
for s in range(10):
    ad, _, _, _ = simulate_visium(seed=s)
    print(f'  seed={s}: {ad.n_obs} spots')

sq.pl.spatial_scatter(adata, color='cluster', img_alpha=0, size=3, shape=None)

## 3. Scoring functions

All six methods: EnrichMap (± GAM), AUCell, scanpy, z-score, GSVA, ssGSEA.

In [ ]:
def score_enrichmap(ad, gene_list, with_gam=False):
    em.tl.score(
        ad, gene_set=gene_list, score_key='_bench',
        smoothing=True, correct_spatial_covariates=with_gam,
    )
    return ad.obs['_bench_score'].values


def score_aucell(ad, gene_list):
    net = pd.DataFrame({'source': '_bench', 'target': gene_list})
    dc.mt.aucell(ad, net=net)
    return dc.pp.get_obsm(
        adata=ad, key='score_aucell'
    ).obsm['score_aucell']['_bench'].values


def score_scanpy(ad, gene_list):
    sc.tl.score_genes(
        ad, gene_list=gene_list, score_name='_bench', use_raw=False,
    )
    return ad.obs['_bench'].values


def score_zscore(ad, gene_list):
    X = ad.X.toarray() if issparse(ad.X) else ad.X
    expr = pd.DataFrame(X.T, index=ad.var_names, columns=ad.obs_names)
    common = [g for g in gene_list if g in expr.index]
    return (expr.loc[common].sum(axis=0) / np.sqrt(len(common))).values


def score_gsva(ad, gene_list):
    X = ad.X.toarray() if issparse(ad.X) else ad.X
    expr = pd.DataFrame(X.T, index=ad.var_names, columns=ad.obs_names)
    result = gp.gsva(
        data=expr, gene_sets={'_bench': gene_list},
        outdir=None, min_size=3,
    )
    es = result.res2d.pivot(index='Term', columns='Name', values='ES')
    return es.loc['_bench'].T.astype(float).reindex(ad.obs_names).values


def score_ssgsea(ad, gene_list):
    X = ad.X.toarray() if issparse(ad.X) else ad.X
    expr = pd.DataFrame(X.T, index=ad.var_names, columns=ad.obs_names)
    result = gp.ssgsea(
        data=expr, gene_sets={'_bench': gene_list},
        outdir=None, sample_norm_method='rank',
        no_plot=True, min_size=3,
    )
    nes = result.res2d.pivot(index='Term', columns='Name', values='NES')
    return nes.loc['_bench'].T.astype(float).reindex(ad.obs_names).values


SCORING_FUNCTIONS = {
    'EnrichMap': lambda ad, gl: score_enrichmap(ad, gl, with_gam=False),
    'EnrichMap (GAM)': lambda ad, gl: score_enrichmap(ad, gl, with_gam=True),
    'AUCell': score_aucell,
    'scanpy': score_scanpy,
    'z-score': score_zscore,
    'GSVA': score_gsva,
    'ssGSEA': score_ssgsea,
}

## 4. Benchmark functions

In [ ]:
def run_concordance(adata, score_dict, labels, n_neighbors=6, max_iter=10):
    """Neighbourhood enrichment concordance for pre-computed scores."""
    adata_tmp = adata.copy()
    sq.gr.spatial_neighbors(
        adata_tmp, n_neighs=n_neighbors,
        coord_type='generic', key_added='spatial',
    )
    W = adata_tmp.obsp['spatial_connectivities']
    row_sums = np.array(W.sum(axis=1)).flatten()
    row_sums[row_sums == 0] = 1.0
    W_norm = W.multiply(1.0 / row_sums[:, None])

    results = {}
    for method, scores in score_dict.items():
        results[method] = {}
        spatial_gt = labels.astype(np.float64).copy()
        for it in range(1, max_iter + 1):
            spatial_gt = W_norm.dot(spatial_gt)
            rho, _ = spearmanr(scores, spatial_gt)
            results[method][it] = rho if not np.isnan(rho) else 0.0
    return results


def run_contamination(
    adata, sig_genes, scoring_fns, clean_scores,
    fractions, n_repeats=3, rng=None,
):
    """Signature contamination robustness."""
    if rng is None:
        rng = np.random.default_rng(0)

    common = [g for g in sig_genes if g in adata.var_names]
    non_sig = [g for g in adata.var_names if g not in common]
    n_sig = len(common)

    results = {m: {f: [] for f in fractions} for m in scoring_fns}

    for frac in fractions:
        n_contam = int(round(n_sig * frac))
        for _ in range(n_repeats):
            noise_genes = list(rng.choice(
                non_sig, n_contam, replace=len(non_sig) < n_contam,
            )) if n_contam > 0 else []
            contam_set = common + noise_genes

            for method, fn in scoring_fns.items():
                try:
                    cs = fn(adata.copy(), contam_set).flatten().astype(np.float64)
                except Exception:
                    cs = np.zeros(adata.n_obs)
                rho, _ = spearmanr(clean_scores[method], cs)
                results[method][frac].append(rho if not np.isnan(rho) else 0.0)

    return results


def run_noise_robustness(
    adata, sig_genes, scoring_fns, clean_scores,
    noise_levels, n_repeats=3, rng=None,
):
    """Noise robustness benchmark."""
    if rng is None:
        rng = np.random.default_rng(0)

    common = [g for g in sig_genes if g in adata.var_names]
    gene_idx = [list(adata.var_names).index(g) for g in common]

    X_dense = adata.X.toarray() if issparse(adata.X) else adata.X
    gene_stds = np.array([X_dense[:, gi].std() for gi in gene_idx])

    results = {m: {lv: [] for lv in noise_levels} for m in scoring_fns}

    for level in noise_levels:
        for _ in range(n_repeats):
            ad_noisy = adata.copy()
            X_n = X_dense.copy()
            for j, gi in enumerate(gene_idx):
                X_n[:, gi] += rng.normal(
                    0, gene_stds[j] * level, X_n.shape[0],
                )
            ad_noisy.X = X_n
            for layer in ad_noisy.layers:
                ad_noisy.layers[layer] = X_n.copy()

            for method, fn in scoring_fns.items():
                try:
                    ns = fn(ad_noisy.copy(), common).flatten().astype(np.float64)
                except Exception:
                    ns = np.zeros(adata.n_obs)
                rho, _ = spearmanr(clean_scores[method], ns)
                results[method][level].append(
                    rho if not np.isnan(rho) else 0.0,
                )

    return results


def run_gene_weight_validation(adata, sig_genes, fold_changes):
    """Correlate inferred weights with true fold changes."""
    if 'gene_contributions' not in adata.uns:
        return np.nan

    score_keys = list(adata.uns['gene_contributions'].keys())
    if not score_keys:
        return np.nan

    contributions = adata.uns['gene_contributions'][score_keys[0]]
    common = [g for g in sig_genes if g in contributions]
    if len(common) < 3:
        return np.nan

    weights = [np.mean(np.abs(contributions[g])) for g in common]
    fc_matched = [fold_changes[sig_genes.index(g)] for g in common]

    rho, _ = spearmanr(weights, fc_matched)
    return rho if not np.isnan(rho) else 0.0


def run_classification(adata, score_dict, labels, n_neighbors=6):
    """
    Classification benchmark: AUROC, AUPRC, F1 (optimal threshold),
    spatial coherence, boundary F1, KS statistic.
    """
    from sklearn.metrics import (
        roc_auc_score, average_precision_score,
        precision_recall_curve, f1_score,
    )
    from scipy.stats import ks_2samp

    # Spatial graph
    adata_tmp = adata.copy()
    sq.gr.spatial_neighbors(
        adata_tmp, n_neighs=n_neighbors,
        coord_type='generic', key_added='spatial',
    )
    W = adata_tmp.obsp['spatial_connectivities']

    # Ground truth Moran's I for spatial coherence normalisation
    y = labels.astype(np.float64)
    n = len(y)
    z = y - y.mean()
    z2 = z @ z
    gt_morans = float((n / W.sum()) * (z @ W.dot(z) / z2)) if z2 > 0 else 0.0

    # Boundary mask
    boundary_mask = np.zeros(n, dtype=bool)
    for i in range(n):
        neighbours = W[i].nonzero()[1]
        if len(neighbours) > 0 and np.any(labels[neighbours] != labels[i]):
            boundary_mask[i] = True

    results = {}
    for method, scores in score_dict.items():
        s = scores.astype(np.float64)
        valid = ~np.isnan(s)
        sv, yv = s[valid], labels[valid]

        if yv.sum() == 0 or yv.sum() == len(yv):
            continue

        rec = {}

        # AUROC
        try:
            rec['AUROC'] = roc_auc_score(yv, sv)
        except ValueError:
            rec['AUROC'] = np.nan

        # AUPRC
        try:
            rec['AUPRC'] = average_precision_score(yv, sv)
        except ValueError:
            rec['AUPRC'] = np.nan

        # Optimal F1
        prec, recall, thresholds = precision_recall_curve(yv, sv)
        prec, recall = prec[:-1], recall[:-1]
        with np.errstate(divide='ignore', invalid='ignore'):
            f1_vals = 2 * (prec * recall) / (prec + recall)
        f1_vals = np.nan_to_num(f1_vals, nan=0.0)
        best_idx = np.argmax(f1_vals)
        rec['F1'] = float(f1_vals[best_idx])
        best_thr = float(thresholds[best_idx])

        # KS statistic
        in_domain = sv[yv == 1]
        out_domain = sv[yv == 0]
        rec['KS'] = ks_2samp(in_domain, out_domain).statistic

        # Spatial coherence (predictions at optimal threshold)
        q = float(np.clip(np.mean(sv <= best_thr), 0.01, 0.99))
        preds = (s > np.quantile(s[valid], q)).astype(float)
        zp = preds - preds.mean()
        zp2 = zp @ zp
        if zp2 > 0 and gt_morans > 0:
            pred_morans = float((n / W.sum()) * (zp @ W.dot(zp) / zp2))
            rec['Spatial coherence'] = float(np.clip(pred_morans / gt_morans, 0, 1))
        else:
            rec['Spatial coherence'] = 0.0

        # Boundary F1
        if boundary_mask.sum() > 0:
            preds_int = preds.astype(int)
            yb = labels[boundary_mask]
            pb = preds_int[boundary_mask]
            if yb.sum() > 0 and yb.sum() < len(yb):
                rec['Boundary F1'] = float(f1_score(yb, pb, zero_division=0))
            else:
                rec['Boundary F1'] = np.nan
        else:
            rec['Boundary F1'] = np.nan

        results[method] = rec

    return results

## 5. Run 100 iterations

In [ ]:
all_classification = []  # (iteration, method, metric, value)

In [ ]:
N_ITERATIONS = 100
CONCORDANCE_ITERS = [1, 3, 5, 7, 10]
CONTAMINATION_FRACS = [0.0, 0.5, 1.0, 2.0, 4.0]
NOISE_LEVELS = [0.0, 0.5, 1.0, 2.0, 3.0]
NOISE_REPEATS = 3
CONTAM_REPEATS = 3

# Collectors
all_concordance = []
all_contamination = []
all_noise = []
all_gene_weights = []

t_start = time.time()

for iteration in tqdm(range(N_ITERATIONS), desc='Simulations'):
    seed = iteration
    rng = np.random.default_rng(seed + 1000)

    # ── Simulate ──────────────────────────────────────────────
    adata, sig_genes, fold_changes, target_label = simulate_visium(
        seed=seed,
    )
    labels = (adata.obs['cluster'] == target_label).astype(int).values

    # ── Score with EnrichMap first (populates gene_contributions)
    em.tl.score(
        adata, gene_set=sig_genes, score_key='_em',
        smoothing=True, correct_spatial_covariates=False,
    )

    # ── Score with all methods ────────────────────────────────
    clean_scores = {}
    for method, fn in SCORING_FUNCTIONS.items():
        try:
            s = fn(adata.copy(), sig_genes).flatten().astype(np.float64)
        except Exception as e:
            print(f'Iter {iteration}, {method} failed: {e}')
            s = np.zeros(adata.n_obs)
        clean_scores[method] = s

    # ── Benchmark 1: Concordance ──────────────────────────────
    conc = run_concordance(
        adata, clean_scores, labels,
        max_iter=max(CONCORDANCE_ITERS),
    )
    for method, iters in conc.items():
        for it in CONCORDANCE_ITERS:
            all_concordance.append({
                'iteration': iteration, 'method': method,
                'smoothing_iter': it, 'concordance': iters[it],
            })

    # ── Benchmark 2: Contamination ────────────────────────────
    cont = run_contamination(
        adata, sig_genes, SCORING_FUNCTIONS, clean_scores,
        CONTAMINATION_FRACS, n_repeats=CONTAM_REPEATS, rng=rng,
    )
    for method, fracs in cont.items():
        for frac, rhos in fracs.items():
            for rho in rhos:
                all_contamination.append({
                    'iteration': iteration, 'method': method,
                    'contamination': frac, 'correlation': rho,
                })

    # ── Benchmark 3: Noise robustness ─────────────────────────
    noise = run_noise_robustness(
        adata, sig_genes, SCORING_FUNCTIONS, clean_scores,
        NOISE_LEVELS, n_repeats=NOISE_REPEATS, rng=rng,
    )
    for method, levels in noise.items():
        for level, rhos in levels.items():
            for rho in rhos:
                all_noise.append({
                    'iteration': iteration, 'method': method,
                    'noise_level': level, 'correlation': rho,
                })

    # ── Benchmark 4: Gene weight validation ───────────────────
    gw_rho = run_gene_weight_validation(
        adata, sig_genes, fold_changes,
    )
    all_gene_weights.append({
        'iteration': iteration, 'weight_fc_rho': gw_rho,
    })

    # ── Benchmark 5: Classification metrics ───────────────────────
    clf = run_classification(adata, clean_scores, labels)
    for method, metrics in clf.items():
        for metric_name, val in metrics.items():
            all_classification.append(
                {
                    "iteration": iteration,
                    "method": method,
                    "metric": metric_name,
                    "value": val,
                }
            )

t_total = time.time() - t_start
print(f'\nCompleted {N_ITERATIONS} iterations in {t_total/60:.1f} minutes')

## 6. Aggregate and save

In [ ]:
df_conc = pd.DataFrame(all_concordance)
df_cont = pd.DataFrame(all_contamination)
df_noise = pd.DataFrame(all_noise)
df_gw = pd.DataFrame(all_gene_weights)
df_clf = pd.DataFrame(all_classification)

df_conc.to_csv(results_dir + 'sim_concordance.csv', index=False)
df_cont.to_csv(results_dir + 'sim_contamination.csv', index=False)
df_noise.to_csv(results_dir + 'sim_noise.csv', index=False)
df_gw.to_csv(results_dir + 'sim_gene_weights.csv', index=False)
df_clf.to_csv(results_dir + "sim_classification.csv", index=False)

print('Results saved.')

## 7. Four-panel figure

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# ── A: Concordance ────────────────────────────────────────────
ax = axes[0, 0]
conc_agg = df_conc.groupby(
    ['method', 'smoothing_iter']
)['concordance'].agg(['mean', 'std']).reset_index()

for method in SCORING_FUNCTIONS.keys():
    sub = conc_agg[conc_agg['method'] == method]
    ax.plot(sub['smoothing_iter'], sub['mean'], 'o-', label=method, ms=4)
    ax.fill_between(
        sub['smoothing_iter'],
        sub['mean'] - sub['std'],
        sub['mean'] + sub['std'],
        alpha=0.15,
    )

ax.set_xlabel('Smoothing iterations')
ax.set_ylabel('Spearman ρ')
ax.set_title('A. Neighbourhood enrichment concordance')
ax.legend(fontsize=6, frameon=False, loc='best')
ax.grid(False)
sns.despine(ax=ax)

# ── B: Contamination ──────────────────────────────────────────
ax = axes[0, 1]
cont_agg = df_cont.groupby(
    ['method', 'contamination']
)['correlation'].agg(['mean', 'std']).reset_index()

for method in SCORING_FUNCTIONS.keys():
    sub = cont_agg[cont_agg['method'] == method]
    ax.plot(sub['contamination'], sub['mean'], 'o-', label=method, ms=4)
    ax.fill_between(
        sub['contamination'],
        sub['mean'] - sub['std'],
        sub['mean'] + sub['std'],
        alpha=0.15,
    )

ax.set_xlabel('Random genes added (× signature size)')
ax.set_ylabel('Spearman ρ (contaminated vs clean)')
ax.set_title('B. Signature contamination robustness')
ax.legend(fontsize=6, frameon=False)
ax.set_ylim(-0.05, 1.05)
ax.grid(False)
sns.despine(ax=ax)

# ── C: Noise robustness ───────────────────────────────────────
ax = axes[1, 0]
noise_agg = df_noise.groupby(
    ['method', 'noise_level']
)['correlation'].agg(['mean', 'std']).reset_index()

for method in SCORING_FUNCTIONS.keys():
    sub = noise_agg[noise_agg['method'] == method]
    ax.plot(sub['noise_level'], sub['mean'], 'o-', label=method, ms=4)
    ax.fill_between(
        sub['noise_level'],
        sub['mean'] - sub['std'],
        sub['mean'] + sub['std'],
        alpha=0.15,
    )

ax.set_xlabel('Noise level (× gene σ)')
ax.set_ylabel('Spearman ρ (noisy vs clean)')
ax.set_title('C. Noise robustness')
ax.legend(fontsize=6, frameon=False)
ax.set_ylim(-0.05, 1.05)
ax.grid(False)
sns.despine(ax=ax)

# ── D: Gene weight validation ─────────────────────────────────
ax = axes[1, 1]
valid_rhos = df_gw['weight_fc_rho'].dropna()

ax.hist(valid_rhos, bins=20, edgecolor='white', alpha=0.7, color='steelblue')
ax.axvline(
    valid_rhos.median(), color='red', ls='--', lw=1.5,
    label=f'Median ρ = {valid_rhos.median():.3f}',
)
ax.set_xlabel('Spearman ρ (inferred weight vs true fold change)')
ax.set_ylabel('Count')
ax.set_title('D. Gene weight validation (100 simulations)')
ax.legend(fontsize=8, frameon=False)
ax.grid(False)
sns.despine(ax=ax)

plt.tight_layout()
plt.savefig(
    figure_dir + 'simulated_benchmarks_100iter.pdf',
    dpi=300, bbox_inches='tight',
)
plt.show()

## 8. Summary statistics

In [ ]:
print('=== Concordance at iteration 7 ===')
conc_7 = df_conc[df_conc['smoothing_iter'] == 7]
print(
    conc_7.groupby('method')['concordance']
    .agg(['mean', 'std']).round(3)
    .sort_values('mean', ascending=False)
    .to_string()
)

print('\n=== Contamination at 2× ===')
cont_2x = df_cont[df_cont['contamination'] == 2.0]
print(
    cont_2x.groupby('method')['correlation']
    .agg(['mean', 'std']).round(3)
    .sort_values('mean', ascending=False)
    .to_string()
)

print('\n=== Noise at 1× ===')
noise_1x = df_noise[df_noise['noise_level'] == 1.0]
print(
    noise_1x.groupby('method')['correlation']
    .agg(['mean', 'std']).round(3)
    .sort_values('mean', ascending=False)
    .to_string()
)

print('\n=== Gene weight validation ===')
v = df_gw['weight_fc_rho'].dropna()
print(f'Median ρ = {v.median():.3f}')
print(f'Mean ρ  = {v.mean():.3f} ± {v.std():.3f}')
print(f'Range   = [{v.min():.3f}, {v.max():.3f}]')

print("\n=== Classification (mean across 100 simulations) ===")
clf_summary = (
    df_clf.groupby(["method", "metric"])["value"].agg(["mean", "std"]).round(3)
)

for metric in ["AUROC", "AUPRC", "F1", "KS", "Spatial coherence", "Boundary F1"]:
    print(f"\n--- {metric} ---")
    sub = clf_summary.xs(metric, level="metric").sort_values("mean", ascending=False)
    print(sub.to_string())